In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os, shutil

# 1. Clean workspace & clone target branch
shutil.rmtree('/kaggle/working/DOC_Extractor_QUAPT', ignore_errors=True)
!git clone -b feature/prd-v1.3-qwen3-vl-multimodal https://github.com/dharicharan/DOC_Extractor_QUAPT.git /kaggle/working/DOC_Extractor_QUAPT

# 2. Install Astral UV
!curl -Ls https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]

# 3. Fast install
%cd /kaggle/working/DOC_Extractor_QUAPT
!uv pip install --system -r requirements.txt
!uv pip install --system -e .
!uv pip install --system qwen-vl-utils transformers accelerate bitsandbytes gradio

print("✅ Setup complete!")

Cloning into '/kaggle/working/DOC_Extractor_QUAPT'...
remote: Enumerating objects: 274, done.
remote: Counting objects: 100% (274/274), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 274 (delta 112), reused 231 (delta 75), pack-reused 0 (from 0)
Receiving objects: 100% (274/274), 1.28 MiB | 6.07 MiB/s, done.
Resolving deltas: 100% (112/112), done.
downloading uv 0.13.7 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/kaggle/working/DOC_Extractor_QUAPT
Using Python 3.13.16 environment at: /usr
Resolved 113 packages in 1.37s                                       
Prepared 6 packages in 1.29s                                             
Uninstalled 1 package in 7ms
Installed 6 packages in 13ms                                
 + av==19.0.2
 + bitsandbytes==0.50.2
 + instructor==1.17.0
 - pillow==13.0.0
 + pillow==12.3.0
 + pypdfium2==5.15.2
 + qwen-vl-utils==0.0.14
Using Python 3.13.16 environment at: /usr
Resolved 63 package

In [ ]:
# (Optional) Pre-download Qwen3-VL-8B weights so the first Extract click is fast.
# Weights are only downloaded here, NOT loaded onto the GPU: app.py loads the model itself,
# and holding a second copy in this kernel would run a 16GB T4 out of memory.
from huggingface_hub import snapshot_download

model_id = "Qwen/Qwen3-VL-8B-Instruct"
print("Downloading weights (cached for app.py)...")
snapshot_download(model_id)
print("Weights cached.")

In [ ]:
# Free any port left from a previous run, then launch the Gradio app (public share link).
# The code was cloned fresh in the setup cell, so no git pull is needed here.
!fuser -k 7860/tcp 2>/dev/null || true

%cd /kaggle/working/DOC_Extractor_QUAPT/src/doc_extractor
!python app.py